# S2 | Confidence Analysis — Human vs Model\n\nConfidence alignment between human self-reported confidence (Likert 1–5 → [0,1]) and model token probability (mean exp(logprob)) under inst-blind conditions.\n\nAll analyses use **variant C** (original question) and the **113-question human study subset**.

In [1]:
import json
import glob
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

ROOT = Path('/home/david/Desktop/yuna/HPA')
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'analysis'))

from utils.constants import MODEL_FAMILY, MODEL_FAMILY_COLORS, GROUP_COLORS
from utils.vqa import VQAAnswerMapper, vqa_accuracy

EOS_TOKENS = {'<|im_end|>', '</s>', '<eos>', '<|endoftext|>', '<|end|>'}
VLM_7B_MODELS = ['Qwen3-VL-8B', 'InternVL-8B', 'LLaVA-1.5-7B', 'LLaVA-Mistral', 'LLaVA-Vicuna']

mapper = VQAAnswerMapper()

In [2]:
# ── Load human confidence (all variants) ──────────────────────────────────────
human_rows = []
for fp in sorted(glob.glob(str(ROOT / 'evaluation/humans/by_participant/*.json'))):
    with open(fp) as f:
        data = json.load(f)
    pid = data.get('code', Path(fp).stem)
    for ans in data.get('answers', []):
        human_rows.append({
            'question_id': ans['question_id'],
            'participant': pid,
            'variant': ans.get('variant', 'C'),
            'confidence': (ans.get('confidence', 3) - 1) / 4.0,
        })

human_df = pd.DataFrame(human_rows)
human_qids = set(human_df['question_id'].unique())
print(f'Human: {human_df["participant"].nunique()} participants, '
      f'{len(human_qids)} questions, variants={sorted(human_df["variant"].unique())}')

Human: 40 participants, 137 questions, variants=['A', 'B', 'C']


In [3]:
# ── Load model confidence + accuracy for VLM 7B models ───────────────────────
VLM_DIR_TO_MODEL = {
    'InternVL3_5-8B':           ('InternVL-8B',   'VLM'),
    'llava-1.5-7b-hf':          ('LLaVA-1.5-7B',  'VLM'),
    'llava-v1.6-mistral-7b-hf': ('LLaVA-Mistral', 'VLM'),
    'llava-v1.6-vicuna-7b-hf':  ('LLaVA-Vicuna',  'VLM'),
    'Qwen3-VL-8B-Instruct':     ('Qwen3-VL-8B',   'VLM'),
}

model_rows = []
for dir_name, (display, group) in VLM_DIR_TO_MODEL.items():
    fpath = ROOT / f'evaluation/logits/vlm/pretrained/{dir_name}/vqa_1k_control_inst_blind.jsonl'
    if not fpath.exists():
        print(f'  [missing] {fpath}')
        continue
    with open(fpath) as f:
        for line in f:
            if not line.strip(): continue
            ex = json.loads(line)
            logits = ex.get('generated_logits', {})
            qid = ex['question_id']
            # variant C only
            ld = logits.get('question')
            if not ld: continue
            probs = [np.exp(t['logprob']) for t in ld['content'] if t['token'] not in EOS_TOKENS]
            if not probs: continue
            answer = ex.get('generated_answers', {}).get('question', '')
            gt = mapper.get_answers(qid)
            acc = vqa_accuracy(answer, gt) if gt else 0.0
            model_rows.append({
                'question_id': qid,
                'model': display,
                'confidence': float(np.mean(probs)),
                'accuracy': acc,
            })

model_df = pd.DataFrame(model_rows)
model_df = model_df[model_df['question_id'].isin(human_qids)].copy()
print(f'Model: {model_df["model"].nunique()} models, {model_df["question_id"].nunique()} questions')

ℹ️  Using local VQA annotations fallback: /home/david/Desktop/yuna/HPA/dataset/vqa/v2_mscoco_val2014_annotations.json
   Loading VQA annotations from /home/david/Desktop/yuna/HPA/dataset/vqa/v2_mscoco_val2014_annotations.json...
   ✓ Loaded 214354 VQA annotations
Model: 5 models, 137 questions


In [4]:
# ── Per-question aggregation ───────────────────────────────────────────────────
hq = (human_df[human_df['variant'] == 'C']
      .groupby('question_id')['confidence']
      .mean().reset_index().rename(columns={'confidence': 'h_conf'}))

mq = (model_df.groupby(['question_id', 'model'])
      .agg(confidence=('confidence', 'mean'), accuracy=('accuracy', 'mean'))
      .reset_index())

merged = mq.merge(hq, on='question_id')
print(f'Merged: {len(merged)} rows')

Merged: 684 rows


## Table 1 — Confidence vs Accuracy: VLM 7B Summary

In [5]:
rows = []
for model_name in VLM_7B_MODELS:
    sub = merged[merged['model'] == model_name]
    if sub.empty: continue
    x, y = sub['confidence'].values, sub['accuracy'].values
    r, p = stats.pearsonr(x, y)
    slope, _ = np.polyfit(x, y, 1)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    rows.append({
        'Model': model_name,
        'n': len(sub),
        'Mean conf.': round(float(x.mean()), 3),
        'Mean acc.': round(float(y.mean()), 3),
        'Slope': round(float(slope), 3),
        'r': round(float(r), 3),
        'p': round(float(p), 4),
        'Sig.': sig,
    })

tbl = pd.DataFrame(rows).set_index('Model')

def color_sig(val):
    if val == 'n.s.': return 'color: #c62828; font-weight: bold'
    return 'color: #2e7d32; font-weight: bold'

(tbl.style
    .background_gradient(subset=['r'], cmap='RdYlGn', vmin=-0.1, vmax=0.6)
    .background_gradient(subset=['p'], cmap='RdYlGn_r', vmin=0, vmax=0.2)
    .background_gradient(subset=['Mean conf.'], cmap='Blues', vmin=0.4, vmax=1.0)
    .background_gradient(subset=['Mean acc.'], cmap='Greens', vmin=0.3, vmax=0.6)
    .background_gradient(subset=['Slope'], cmap='PuOr', vmin=0, vmax=2.0)
    .applymap(color_sig, subset=['Sig.'])
    .format({'r': '{:.3f}', 'p': '{:.4f}', 'Mean conf.': '{:.3f}',
             'Mean acc.': '{:.3f}', 'Slope': '{:.3f}'})
    .set_caption('VLM 7B: Confidence vs Accuracy (inst-blind, variant C)')
)

AttributeError: 'Styler' object has no attribute 'applymap'

## Table 2 — Human vs Model Confidence Correlation by Entity Group

In [ ]:
sem = pd.read_json(ROOT / 'dataset/vqa/vqa1k_semantics.jsonl', lines=True)[['question_id','ent','op']]
ENT_MAP = {'vehicle':'visual-other','product':'visual-other','place':'visual-other','other':'text'}
sem['ent_g'] = sem['ent'].map(lambda x: ENT_MAP.get(x, x))
merged_sem = merged.merge(sem, on='question_id', how='left')

ent_values = ['object', 'person', 'animal', 'food', 'visual-other', 'text']
ent_rows = []
for model_name in VLM_7B_MODELS:
    row = {'Model': model_name}
    for ent in ent_values:
        sub = merged_sem[(merged_sem['model'] == model_name) & (merged_sem['ent_g'] == ent)]
        if len(sub) < 5:
            row[ent] = float('nan')
        else:
            r, _ = stats.pearsonr(sub['h_conf'], sub['confidence'])
            row[ent] = round(r, 3)
    ent_rows.append(row)

ent_tbl = pd.DataFrame(ent_rows).set_index('Model')

(ent_tbl.style
    .background_gradient(cmap='RdYlGn', vmin=-0.4, vmax=0.6, axis=None)
    .format('{:.3f}', na_rep='—')
    .set_caption('r(human conf, model conf) by Entity Group — VLM 7B (inst-blind, variant C)')
)

,object,person,animal,food,visual-other,text
Model,,,,,,
Qwen3-VL-8B,0.082,-0.152,-0.003,-0.452,0.019,0.546
InternVL-8B,0.443,0.292,-0.004,0.575,0.006,0.357
LLaVA-1.5-7B,-0.038,-0.176,-0.166,0.216,0.433,0.261
LLaVA-Mistral,-0.141,0.002,-0.153,-0.016,0.171,0.477
LLaVA-Vicuna,-0.123,-0.352,-0.180,0.131,0.325,-0.044


## Table 3 — Human vs Model Confidence Correlation by Operation Type

In [ ]:
OP_MAP = {'know':'other','text':'other','temp':'other','comp':'other','other':'other','cause':'other'}
sem['op_g'] = sem['op'].map(lambda x: OP_MAP.get(x, x))
merged_sem2 = merged.merge(sem, on='question_id', how='left')

op_values = ['attr', 'count', 'ident', 'spat', 'exist', 'act', 'other']
op_rows = []
for model_name in VLM_7B_MODELS:
    row = {'Model': model_name}
    for op in op_values:
        sub = merged_sem2[(merged_sem2['model'] == model_name) & (merged_sem2['op_g'] == op)]
        if len(sub) < 5:
            row[op] = float('nan')
        else:
            r, _ = stats.pearsonr(sub['h_conf'], sub['confidence'])
            row[op] = round(r, 3)
    op_rows.append(row)

op_tbl = pd.DataFrame(op_rows).set_index('Model')

(op_tbl.style
    .background_gradient(cmap='RdYlGn', vmin=-0.4, vmax=0.6, axis=None)
    .format('{:.3f}', na_rep='—')
    .set_caption('r(human conf, model conf) by Operation Type — VLM 7B (inst-blind, variant C)')
)

,attr,count,ident,spat,exist,act,other
Model,,,,,,,
Qwen3-VL-8B,0.286,-0.201,-0.099,0.327,0.534,0.456,-0.293
InternVL-8B,0.020,0.321,0.572,0.425,-0.116,0.296,0.287
LLaVA-1.5-7B,0.213,-0.002,0.615,0.460,0.285,-0.112,-0.018
LLaVA-Mistral,0.184,-0.217,0.459,0.255,0.321,0.480,0.376
LLaVA-Vicuna,0.033,-0.193,0.645,0.449,0.580,0.106,-0.296
